![Nextia Learning · SQL and Data Preparation for AI](https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/c04-lesson-notebooks-example/nextia-notebook-branding/assets/courses/c04-header.png)

# Join tables

**[C04-M02-L03 · Join tables](https://learning.nextia-ai.com/courses/sql/m02/join-tables/)** · C04, SQL and Data Preparation for AI · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)


**You will learn to** combine tickets with customers and with outcome events using inner and left joins, count the rows before and after each join, find the rows with no match in both directions, and see how a join repeats rows.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | `GROUP BY`, `COUNT(DISTINCT …)` and `CASE`, from [Aggregate correctly](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/). The keys that connect the tables, from [Relational concepts](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/). |
| **You do not need** | The local project from Module 1. The setup below builds the same database here. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m02/join-tables/) has the full explanations, the diagrams, the interactive explorer and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M02-L03-join-tables/join-tables-solution.ipynb).

## Setup 1 of 2: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`, in the folder `c04-data`. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `c04-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db.
# Standard library only. The same steps as get_data.py in Module 1.
import csv, hashlib, sqlite3, subprocess, urllib.request
from pathlib import Path

BASE = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/data/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
DATA = Path("c04-data")
DB = DATA / "larkfield.db"

def fetch(name, expected):
    path = DATA / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        DATA.mkdir(exist_ok=True)
        try:
            urllib.request.urlretrieve(BASE + name, path)
        except OSError:
            # Python without certificates (python.org on macOS): try curl.
            if subprocess.run(["curl", "-fsSL", "-o", str(path), BASE + name]).returncode:
                raise SystemExit(f"Could not download {name}. Check your internet connection.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder c04-data and run again.")
    return path

if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup 2 of 2: two helpers

The next cell defines two small functions. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
import hashlib
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        display(pd.DataFrame(shown, columns=columns))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    got_columns, got = run(query, db)
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")
        return True
    return False

![](https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/c04-lesson-notebooks-example/nextia-notebook-branding/assets/section-divider-light.png)

## 1. Four tickets and five events

A **join** combines two tables into one result. For each row of the first (left) table, it finds the rows of the second (right) table that match a condition, and puts each matching pair side by side in one row. The condition comes after `ON`. It usually says that a foreign key equals a primary key.

The next cell makes two tiny tables in memory, `tiny`, so that you can see every row:

- **tickets**: T-1 (customer C-1), T-2 (C-1), T-3 (C-2) and T-4 (no customer).
- **outcomes**: O-1 and O-2 for T-1, O-3 for T-2, O-4 for T-3, and O-5 for **T-9**, a ticket that is not in `tickets`.

T-1 has **two** events. T-4 has **no** event. O-5 is an **orphan**: its key points to nothing.

Run the cell. It prints nothing.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE customers (customer_id TEXT, segment TEXT);
INSERT INTO customers VALUES ('C-1', 'home'), ('C-2', 'trade'), ('C-3', 'business');
CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, team TEXT);
INSERT INTO tickets VALUES ('T-1', 'C-1', 'delivery'), ('T-2', 'C-1', 'returns'),
    ('T-3', 'C-2', 'payment'), ('T-4', NULL, 'account');
CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, status TEXT);
INSERT INTO outcomes VALUES ('O-1', 'T-1', 'escalated'), ('O-2', 'T-1', 'resolved'),
    ('O-3', 'T-2', 'resolved'), ('O-4', 'T-3', 'resolved'), ('O-5', 'T-9', 'resolved');
""");

An **inner join** keeps only the pairs that match.

> **Predict.** The left table has 4 rows. How many rows does this inner join return, and which tickets are in it? Write it down, then run the cell.

In [ ]:
sql("""
SELECT t.ticket_id, t.team, o.outcome_id, o.status
FROM tickets AS t
INNER JOIN outcomes AS o ON o.ticket_id = t.ticket_id
ORDER BY t.ticket_id, o.outcome_id;
""", db=tiny)

> **Check.** You should see **4 rows**: T-1 twice (it matched two events) and no T-4 (it matched nothing). The count is the same as the left table, but the rows are not: one ticket is lost and one is repeated, and the two changes hide each other.

A **left join** keeps every row of the left table. A left row with no match gets one row with NULL in the right table's columns.

> **Predict.** How many rows now? Then run the cell.

In [ ]:
sql("""
SELECT t.ticket_id, t.team, o.outcome_id, o.status
FROM tickets AS t
LEFT JOIN outcomes AS o ON o.ticket_id = t.ticket_id
ORDER BY t.ticket_id, o.outcome_id;
""", db=tiny)

> **Check.** You should see **5 rows**. T-4 is there with `None` (NULL) for `outcome_id` and `status`. O-5 is in neither result: a left join keeps the unmatched rows of the **left** table only.

**The rule:** each left row gives one result row per match. One match keeps the count. Zero matches drop the row (inner) or keep it with NULLs (left). Two or more matches repeat the row.

So count the **distinct** keys as well as the rows. Run the cell: the inner join has 3 tickets, the left join 4.

In [ ]:
sql("""
SELECT
    (SELECT COUNT(DISTINCT t.ticket_id) FROM tickets AS t
       INNER JOIN outcomes AS o ON o.ticket_id = t.ticket_id) AS inner_tickets,
    (SELECT COUNT(DISTINCT t.ticket_id) FROM tickets AS t
       LEFT JOIN outcomes AS o ON o.ticket_id = t.ticket_id) AS left_tickets;
""", db=tiny)

## 2. Worked example: count the real joins

Mei wants each ticket with its customer's segment and its outcome events. Before she writes a report, she counts the rows of each join. Each query in brackets gives one number. Run the cell.

In [ ]:
sql("""
SELECT
    (SELECT COUNT(*) FROM tickets) AS ticket_rows,
    (SELECT COUNT(*) FROM tickets AS t
       INNER JOIN customers AS c ON c.customer_id = t.customer_id) AS inner_customers,
    (SELECT COUNT(*) FROM tickets AS t
       LEFT JOIN customers AS c ON c.customer_id = t.customer_id) AS left_customers,
    (SELECT COUNT(*) FROM tickets AS t
       INNER JOIN outcomes AS o ON o.ticket_id = t.ticket_id) AS inner_outcomes,
    (SELECT COUNT(*) FROM tickets AS t
       LEFT JOIN outcomes AS o ON o.ticket_id = t.ticket_id) AS left_outcomes;
""")

> **Check.** You should see `1417  1396  1417  1893  1901`. Mei explains each change before she uses a join:

- **Tickets and customers:** the left join returns 1,417, the same as `tickets`, so no ticket matched two customers. The inner join returns 1,396: **21 ticket rows have no customer.**
- **Tickets and outcomes:** the joins return more rows than `tickets`, because one ticket can have many events. The grain changes from "one ticket" to "one event of one ticket".

Find the 21 tickets with no customer: a left join, then `IS NULL` on the right table's key. Run the cell.

In [ ]:
sql("""
SELECT t.customer_id, COUNT(*) AS ticket_rows
FROM tickets AS t
LEFT JOIN customers AS c ON c.customer_id = t.customer_id
WHERE c.customer_id IS NULL
GROUP BY t.customer_id;
""")

> **Check.** You should see 5 rows: `None` (NULL) with 12 rows, which are guest tickets, and C-0241 to C-0244 with 9 rows together, which are deleted accounts. 12 + 9 = 21. An inner join drops these real tickets without a word.

The other direction: events with no ticket. This join starts from `outcomes`, so it keeps every event.

> **Predict.** How many events have no ticket? Then run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS event_rows
FROM outcomes AS o
LEFT JOIN tickets AS t ON t.ticket_id = o.ticket_id
WHERE t.ticket_id IS NULL;
""")

> **Check.** You should see **4**: the events of T-9990 to T-9993. No join that starts from `tickets` shows them.

> **Your turn: change one thing.** Copy the query into the next cell. Start from `tickets`, left join `outcomes`, keep the rows where `o.outcome_id IS NULL`, and count them. You should see **8**: tickets with no event yet.

In [ ]:
sql("""
-- Tickets with no outcome event. Write the query here.
""")

## 3. Misconception: "a join only adds columns"

That is true only when each left row matches **at most one** right row. If the key is not unique on the right, rows repeat. If it is not unique on **both** sides, the repeats multiply.

Ticket T-11149 appears twice in `tickets` (an export page overlap) and has 5 rows in `outcomes`, one of them a repeated event.

> **Predict.** How many rows does the join give for T-11149? Then run the cell.

In [ ]:
sql("""
SELECT t.ticket_id, o.outcome_id, o.recorded_at, o.status
FROM tickets AS t
INNER JOIN outcomes AS o ON o.ticket_id = t.ticket_id
WHERE t.ticket_id = 'T-11149'
ORDER BY o.outcome_id;
""")

> **Check.** You should see **10 rows**: 2 ticket rows × 5 event rows. Add `DISTINCT` after `SELECT` in the cell above and run it again: 5 rows. That is still one too many, because the repeated event (O-01569 repeats O-01568) has its own ID. T-11149 has 4 real events.

After a join with `outcomes`, count **distinct tickets** (`COUNT(DISTINCT t.ticket_id)`), never rows. Module 4 removes both kinds of duplicate in a cleaning step.

## 4. Failure case: a join with no condition

Tomás forgets the `ON` line.

> **Predict.** What does this count mean? Then run the cell.

In [ ]:
sql("SELECT COUNT(*) AS result_rows FROM tickets JOIN outcomes;")

> **Check.** You should see **2,655,458** = 1,417 × 1,874. Without a condition, a join matches every row with every row: a **cross join**. It looks like "a lot of events", and it gives no error.

> **Warning.** Do not run a cross join of three tables, such as `FROM tickets AS a, tickets AS b, tickets AS c`: it asks for about 2.8 billion rows and does not finish. If you start one by mistake, stop it: in Colab **Runtime › Interrupt execution**, in Kaggle and Jupyter the **■ stop** button.

**Diagnose:** compare the row count after a join with the counts before it. A result close to `rows × rows` means a missing condition. **Fix:** give every join an `ON` condition on the key. The quiet case is worse: a wrong key, such as `customer_id` instead of `ticket_id`, still finishes and gives every ticket the events of all the customer's tickets.

## 5. Your turn: list the orphan events

> **Your turn.** Write a query that lists the events whose ticket is not in `tickets`: `outcome_id`, `ticket_id` and `status`, sorted by `outcome_id`. The starting query lists all 1,874 events; add a join and a `WHERE` line. Then run the check cell. When it is correct, you see 4 rows and `Check passed.`

On your computer, save the query as `queries/25-orphan-events.sql` in your project, as in the lesson.

In [ ]:
orphan_events = """
-- Events whose ticket is not in tickets.
SELECT o.outcome_id, o.ticket_id, o.status
FROM outcomes AS o
-- add the join and the condition here
ORDER BY o.outcome_id;
"""
sql(orphan_events)

In [ ]:
check(orphan_events, rows=4, columns=3, expected="283306aff5036616", ordered=True)

## 6. Change one condition: customers with no ticket

Omar asks: "Which segments do the customers with no ticket belong to?" Write a query that starts from `customers`, finds the customers with no ticket, and counts them per `segment`, as the segment is stored (do not change its spelling). Two columns: the segment and the count.

> **Predict.** What is the total? You know it from the lesson.

In [ ]:
quiet_customers = """
-- Customers with no ticket, per segment.
SELECT c.segment, COUNT(*) AS customers_without_tickets
FROM customers AS c
-- add the join and the condition here
GROUP BY c.segment;
"""
sql(quiet_customers)

In [ ]:
check(quiet_customers, rows=4, columns=2, expected="78db5de27e177cc9")

The total is **46**. `Trade` with a capital T is the messy category from Module 1: it is a separate group until Module 4 cleans it.

## 7. Module practice: the segment report

The support team wants **ticket volume and resolution outcomes by customer segment**. The report counts **tickets**, not rows, and keeps the tickets that have no customer. Five columns, in this order:

1. the customer group: `home`, `trade`, `business`, `guest` (no `customer_id`) or `unknown` (a `customer_id` that is not in `customers`)
2. the number of tickets
3. the tickets with at least one `resolved` event
4. the tickets with at least one `escalated` event
5. the tickets with no event.

The starting query runs, and its numbers look plausible, but they are wrong. Fix it. When it is correct, you see 5 rows, the tickets add up to 1,403, and the check passes. On the lesson page, the worked example on the tiny tables shows each step.

> **Warning.** Do not name the `CASE` result `segment`. SQLite then groups by the column `c.segment`, and gives no error. Use `customer_group`.

In [ ]:
segment_report = """
-- Ticket volume and outcomes by customer group.
-- Count distinct tickets. Keep the tickets with no customer.
SELECT
    c.segment AS customer_group,
    COUNT(*) AS tickets
FROM tickets AS t
INNER JOIN customers AS c ON c.customer_id = t.customer_id
INNER JOIN outcomes AS o ON o.ticket_id = t.ticket_id
GROUP BY customer_group;
"""
sql(segment_report)

In [ ]:
check(segment_report, rows=5, columns=5, expected="051a4e34c3c4b294")

**Reconcile** your report with the source tables before you trust it. Run the cell: it adds up your `tickets` column and counts the distinct tickets in `tickets`. The two numbers must be equal.

In [ ]:
columns, rows = run(segment_report)
print("report total:", sum(row[1] for row in rows))
print("source total:", con.execute("SELECT COUNT(DISTINCT ticket_id) FROM tickets").fetchone()[0])

Then explain the differences, in your own words, before you open the model answer on the lesson page:

- Why is the `resolved` column 4 lower than the number of distinct tickets with a `resolved` event in `outcomes`?
- Which records have no match, in each direction, and how many of each?

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m02/join-tables/#module-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that an inner join drops rows with no match, that a left join keeps every left row, and that a key that is not unique repeats rows. You count the rows and the distinct keys before and after each join, and you find the unmatched rows in both directions.

![](https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/c04-lesson-notebooks-example/nextia-notebook-branding/assets/section-divider-light.png)

**Next lesson:** [Readable queries](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m02/join-tables/)

![Nextia Learning · learning.nextia-ai.com](https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/c04-lesson-notebooks-example/nextia-notebook-branding/assets/notebook-footer-light.png)

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.